# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import time
DEMO_START_TIME = time.perf_counter()

In [2]:
%pip -q install "transformers==4.57.6" "datasets" "accelerate" "scikit-learn" "pandas"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [4]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [5]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [6]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [7]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [8]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [9]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [10]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Linear(in_features=768, out_features=2, bias=True)


In [12]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=6,             # завдання 0.1 №9: порівнюємо 3 та 6 епох
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_strategy="epoch",      # бачимо loss після кожної епохи
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [14]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.527500,0.368141,0.838235,0.885813
2,0.314300,0.361860,0.833333,0.887043
3,0.157500,0.439773,0.879902,0.912656
4,0.081300,0.548645,0.867647,0.905594
5,0.039600,0.636767,0.867647,0.904594
6,0.020800,0.689757,0.855392,0.897391


{'eval_loss': 0.6897570490837097,
 'eval_accuracy': 0.8553921568627451,
 'eval_f1': 0.8973913043478261,
 'eval_runtime': 0.4937,
 'eval_samples_per_second': 826.392,
 'eval_steps_per_second': 14.178,
 'epoch': 6.0}

## 4. Перевірка на власних прикладах

In [15]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 99.9%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 0.2%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.9%



### Додаткове завдання №9 — результати після 6 епох

У демо кількість епох збільшено з **3 до 6**. Після запуску клітинки покажуть фактичні `eval_accuracy`, `eval_f1`, `eval_loss` для кожної епохи. Порівнюємо показники після третьої та шостої епох, а не припускаємо, що довше навчання завжди краще.

In [16]:
import pandas as pd

history = pd.DataFrame(trainer.state.log_history)

epochs = history.dropna(subset=["eval_loss"])[
    ["epoch", "eval_loss", "eval_accuracy", "eval_f1"]
]

epochs = epochs.drop_duplicates(subset=["epoch"], keep="last")

display(epochs.round(4))

,epoch,eval_loss,eval_accuracy,eval_f1
1,1.0,0.3681,0.8382,0.8858
3,2.0,0.3619,0.8333,0.8870
5,3.0,0.4398,0.8799,0.9127
7,4.0,0.5486,0.8676,0.9056
9,5.0,0.6368,0.8676,0.9046
13,6.0,0.6898,0.8554,0.8974


### Додаткове завдання №10 — ще чотири власні пари

Нижче дві пари, що мають однаковий зміст, і дві — з різним змістом. Прогноз та впевненість записуються тільки після реального запуску.

In [17]:
own_pairs = [
    ("A student completed the assignment on Monday.", "The student finished the task on Monday."),
    ("The train arrived two hours late.", "The train was delayed by two hours."),
    ("The meeting starts at nine in the morning.", "The meeting was cancelled yesterday."),
    ("The package was delivered to the office.", "The parcel has not yet been shipped."),
]
for first, second in own_pairs:
    print(f"1: {first}\n2: {second}\n→ {predict(model, first, second)}\n")

1: A student completed the assignment on Monday.
2: The student finished the task on Monday.
→ парафраз: 99.9%

1: The train arrived two hours late.
2: The train was delayed by two hours.
→ парафраз: 99.1%

1: The meeting starts at nine in the morning.
2: The meeting was cancelled yesterday.
→ парафраз: 98.8%

1: The package was delivered to the office.
2: The parcel has not yet been shipped.
→ парафраз: 99.5%



---
## Фактичний час виконання демо

Це приблизний час від стартової клітинки до цієї клітинки **в межах поточної Colab-сесії**. Якщо були паузи, ручне очікування чи перезапуски, запиши це у відповіді окремо. Час завантаження GPU-моделей та швидкість інтернету можуть відрізнятися.

In [18]:
print(f"Від старту демо: {(time.perf_counter() - DEMO_START_TIME)/60:.2f} хв")

Від старту демо: 5.34 хв


---
## Відповіді на питання 0.1: 1–10

**1. Як довго виконувалися клітинки `01_bert_head`? Чи були проблеми?**  
Загальний час виконання ноутбука становив 5,34 хвилини (приблизно 5 хв 20 с), з яких близько 2 хв було витрачено на підключення залежностей, а саме тренування моделі зайняло 3 хв 10 с.

Під час встановлення бібліотек виникло попередження про конфлікт версій залежностей huggingface-hub, gradio та diffusers. Також було попередження про відсутність HF_TOKEN. Проте це не завадило завантаженню моделі, тренуванню та виконанню всіх клітинок.

**2. Що таке `[CLS]` і як він працює?**  
`[CLS]` — спеціальний токен на початку послідовності BERT. Його векторне представлення під час проходження через attention взаємодіє з іншими токенами й використовується для отримання ознак усього вхідного тексту. Класифікаційна голова перетворює ці ознаки на оцінки класів. Сам токен не містить «готового підсумку» ще до проходження шарів моделі.

**3. Для чого спочатку створювали `bert-base-uncased`?**  
Це базова англомовна BERT-модель, попередньо навчена насамперед на відновленні замаскованих токенів (*Masked Language Modeling*) та задачі зв'язку між реченнями (*Next Sentence Prediction*). Її можна адаптувати до класифікації, пошуку відповідей, визначення сутностей тощо. `uncased` означає, що текст приводиться до нижнього регістру.

**4. Чи варто боротися з проявами сексизму мовних моделей?**  
Так. Якщо модель пов'язує професії або здібності з гендером, це може відтворювати несправедливі стереотипи. Потрібно тестувати моделі на контрастних прикладах, оцінювати відмінності між групами, поліпшувати й балансувати дані та контролювати наслідки використання. Приклади `man/woman` лише показують можливі асоціації; за двома реченнями не можна повністю оцінити упередженість.

**5. Що означають 0 і 1 під час токенізації пари речень?**  
У `token_type_ids` **0** позначає токени першого речення (і службові токени поруч із ним), **1** — другого. Так BERT розрізняє сегменти, хоча обидва речення подаються однією послідовністю.

**6. Що означають числа 768 і 2?**  
**768** — розмір прихованого представлення токена в BERT Base (`hidden_size`); **2** — кількість вихідних класів задачі MRPC (парафраз / не парафраз) у новій класифікаційній голові.

**7. Чому у backbone значно більше параметрів, ніж у голові? Це добре?**  
Так, це очікувано. Великий backbone уже містить знання про мову, а маленька голова лише перетворює його ознаки на оцінки двох класів. У цьому демо оновлюються **і backbone, і голова**, тому обчислення все ще потребують ресурсів; з маленьким датасетом треба стежити за перенавчанням.

**8. Що вимірюють Accuracy та F1 у `01_bert_head`?**  
`Accuracy` — частка правильно класифікованих пар речень у validation. `F1` — гармонійне середнє precision та recall для позитивного класу `label=1` (речення є парафразами). F1 корисна, коли одного відсотка правильних відповідей недостатньо для оцінки позитивного класу.

**9. Що відбулося після збільшення кількості епох?**  
Я збільшив кількість епох тренування з 3 до 6. Найкращі показники Accuracy та F1 були отримані після третьої епохи.

| Метрика | 3 епоха | 6 епоха |
|---|---:|---:|
| Training Loss | 0,1575 | 0,0208 |
| Validation Loss | 0,4398 | 0,6898 |
| Accuracy | 87,99% | 85,54% |
| F1 | 0,9127 | 0,8974 |

Незважаючи на те, що помилка на тренувальних даних продовжувала зменшуватися, помилка на валідаційних даних зростала, а Accuracy та F1 після третьої епохи погіршилися.

Це свідчить про перенавчання моделі (overfitting). Модель почала надто сильно пристосовуватися до тренувальних даних, через що погіршилася її здатність узагальнювати інформацію на нових прикладах. За результатами цього експерименту третя епоха є найкращою за Accuracy та F1.

**10. Чотири власні перевірки.**  
Я додав чотири власні пари речень: дві пари з однаковим змістом та дві з різним.

Модель правильно визначила 2 із 4 прикладів. Для справжніх парафразів вона видала ймовірності 99,9% та 99,1%. Проте для речень із різним змістом модель також видала високі ймовірності парафразу — 98,8% та 99,5%.

Наприклад, речення «The package was delivered to the office» та «The parcel has not yet been shipped» мають протилежний зміст, але модель визначила їх як парафраз із ймовірністю 99,5%.

Можливо, модель надто сильно орієнтується на схожість слів і загальної тематики речень, не враховуючи відмінності в їхньому змісті. Також на результат могло вплинути перенавчання після збільшення кількості епох.